[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week7_cicd_for_ml_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 7 -- CI/CD for Machine Learning (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** Great Expectations data tests, model behavioural tests, GitHub Actions CI/CD, staging environment

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Write three Great Expectations data tests for CreditDefault-NG training data
- Write three model behavioural tests encoding domain knowledge about credit default
- Build a GitHub Actions workflow that blocks deployment on any test failure
- Distinguish staging from production environment and explain why both must exist
- Block a deliberate deployment failure with the CI/CD pipeline and then fix it



---

## MONDAY -- "Data Tests with Great Expectations"


Great Expectations defines assertions about data that must be true before the model trains. For CreditDefault-NG: all 23 required columns are present, PAY_0 values are in {-2,-1,0,1,2,3,4,5,6,7,8} (valid bureau codes), LIMIT_BAL is positive, no null values in the target column, and the class balance is between 10% and 40%.

Pause and Predict -- write three data tests you think are most important before reading the code. Which failure mode is most dangerous for the credit model?


### Exercise 7.1 -- Great Expectations suite

Build the suite: schema check (18 required columns), PAY_0 range (-2 to 8), LIMIT_BAL positive, null check on DEFAULT_NEXT_MONTH, class balance 10%-40%. Run on clean training data (should pass) and on a corrupted file (should fail).


In [ ]:
# Exercise 7.1: Great Expectations suite
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: Data Tests with Great Expectations --
# Great Expectations data tests
train = pd.read_csv('datasets/creditdefault_training.csv')

# Schema test: all required features must be present
required = ['LIMIT_BAL','PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6',
            'BILL_AMT1','BILL_AMT2','BILL_AMT3',
            'PAY_AMT1','PAY_AMT2','PAY_AMT3',
            'SEX','EDUCATION','MARRIAGE','AGE','DEFAULT_NEXT_MONTH']
for col in required:
    assert col in train.columns, f'Missing required column: {col}'

# Range checks
assert train['LIMIT_BAL'].min() > 0, 'LIMIT_BAL must be positive'
assert train['PAY_0'].isin(range(-2, 9)).all(), 'Invalid PAY_0 code'

# No nulls in target
assert train['DEFAULT_NEXT_MONTH'].isnull().sum() == 0, 'Null target values'

# Class balance: default rate between 10% and 40%
default_rate = train['DEFAULT_NEXT_MONTH'].mean()
assert 0.10 <= default_rate <= 0.40, f'Extreme class balance: {default_rate:.1%}'
print('All data tests passed')


### Expected Output (illustrative — Great Expectations)

```
Validation Results: 4/4 expectations passed (clean data)
Validation Results: 2/4 expectations passed (corrupted data)
  FAILED: expect_column_values_to_be_in_set (PAY_0 contains value 12, outside {-2..8})
  FAILED: expect_column_values_to_not_be_null (DEFAULT_NEXT_MONTH has 3 nulls)
```
Run the same suite against both the clean training data and a deliberately corrupted copy —
if the suite passes on both, the checks aren't actually testing anything.


### Monday Morning Moment

*Slack -- Monday, 11:00am.*

**Sola Fashola:** What tests are you writing first?

**Temi Adeyemi:** Data tests. Schema, null checks, PAY_0 range check.

**Sola Fashola:** Most dangerous data test failure for the credit model?

**Temi Adeyemi:** The PAY_0 range check. If the bureau adds a new delinquency code outside our expected range, the model receives an input it has never seen -- and fails silently.

**Sola Fashola:** That is the test that would have caught the August bureau schema change before it reached production. Write it first.

**Dr. Emeka Obi:** And a behavioural test. If the model ever predicts that a customer who missed two payments is less likely to default than one who paid on time, something fundamental is wrong.




---

## TUESDAY -- "Model Behavioural Tests"


Behavioural tests encode domain knowledge about what a correct credit model must do. Three tests: (1) a customer with maximum payment delay (PAY_0=8) must score higher default probability than a current customer (PAY_0=-1); (2) a customer with zero credit utilisation must score lower than a maxed-out customer; (3) the model's performance floor must hold on the holdout set.


### Exercise 7.2 -- Three behavioural tests

Write three directional tests: (a) higher PAY_0 -> higher default probability, (b) lower PAY_AMT1 relative to BILL_AMT1 -> higher probability, (c) AGE < 25 with PAY_0 > 0 -> higher than AGE > 50 with PAY_0 = -1.


In [ ]:
# Exercise 7.2: Three behavioural tests
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: Model Behavioural Tests --
import pytest, time, numpy as np
from sklearn.metrics import roc_auc_score
import mlflow.sklearn, pandas as pd

model   = mlflow.sklearn.load_model('models:/credit-default-v2/Staging')
holdout = pd.read_csv('datasets/creditdefault_holdout.csv')
X = holdout.drop('DEFAULT_NEXT_MONTH', axis=1)
y = holdout['DEFAULT_NEXT_MONTH']

def test_performance_floor():
    auc = roc_auc_score(y, model.predict_proba(X)[:, 1])
    assert auc >= 0.80, f'AUC {auc:.4f} below 0.80 SLA floor'

def test_payment_delay_increases_default_risk():
    """A customer with maximum delay must score higher than a current payer."""
    base   = X.iloc[[0]].copy()
    late   = base.assign(PAY_0=8)   # 8 months delay
    ontime = base.assign(PAY_0=-1)  # paid on time
    p_late   = model.predict_proba(late)[:, 1][0]
    p_ontime = model.predict_proba(ontime)[:, 1][0]
    assert p_late > p_ontime, f'Delayed payer ({p_late:.4f}) not riskier than on-time ({p_ontime:.4f})'

def test_high_utilisation_increases_default_risk():
    """Maxed-out credit must score higher than zero utilisation."""
    base   = X.iloc[[0]].copy()
    limit  = float(base.LIMIT_BAL.iloc[0])
    maxed  = base.assign(BILL_AMT1=limit)
    unused = base.assign(BILL_AMT1=0.0)
    p_maxed  = model.predict_proba(maxed)[:, 1][0]
    p_unused = model.predict_proba(unused)[:, 1][0]
    assert p_maxed > p_unused, 'Maxed-out not riskier than zero utilisation'


### Expected Output (illustrative — pytest)

```
tests/model/test_model.py::test_pay0_directionality PASSED
tests/model/test_model.py::test_payment_ratio_directionality PASSED
tests/model/test_model.py::test_age_pay0_interaction PASSED
3 passed in 1.42s
```
These three tests encode facts a domain expert knows and a metrics dashboard can't see — a
model with AUC=0.85 that fails `test_pay0_directionality` is a model making a specific,
identifiable mistake despite a healthy-looking aggregate score.



---

## WEDNESDAY -- "GitHub Actions Workflow"


GitHub Actions runs the test suite on every PR to main. Data tests run first. If data tests fail, model tests do not run -- fail fast: a model test on corrupt data is meaningless.


### Exercise 7.3 -- GitHub Actions workflow

Write the complete workflow. data_tests runs first. model_tests depends on data_tests. Add a third job: generate and upload the Evidently drift report as a workflow artifact.


In [ ]:
# Exercise 7.3: GitHub Actions workflow
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: GitHub Actions Workflow --
# .github/workflows/ml_tests.yml
name: ML Tests
on:
  pull_request:
    branches: [main]
jobs:
  data_tests:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with: {python-version: '3.10'}
      - run: pip install -r requirements.txt
      - run: dvc pull datasets/
      - run: python -m pytest tests/data/ -v
  model_tests:
    needs: data_tests  # only runs if data_tests passes
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with: {python-version: '3.10'}
      - run: pip install -r requirements.txt
      - run: dvc pull datasets/ models/
      - run: python -m pytest tests/model/ -v


### Expected Output (illustrative — GitHub Actions run)

```
✓ data_tests        (18s)
✓ model_tests        (12s)  -- depends on: data_tests
✓ upload-drift-report (4s)
All checks have passed
```
If `data_tests` fails, `model_tests` should show as **skipped**, not failed and not run — fail
fast on the cheaper, more fundamental check before spending time on the expensive one.



---

## THURSDAY -- "Staging vs Production"


Staging is production-equivalent: same hardware, same feature store, same monitoring. A model in Staging receives mirrored real traffic but its predictions are logged rather than served. One week of shadow mode in staging is required before Production promotion.


### STOP AND TRACE

Before running:

@pytest.fixture(scope='module')
def model():
    return mlflow.sklearn.load_model('models:/credit-default-v2/Staging')

What does scope='module' mean?
If loading the model takes 3 seconds and you have 20 tests, what is the total load time with scope='module' vs the default scope='function'?
Why this matters: scope='function' reloads the model for every test. With 20 tests and a 3-second load, that is 60 seconds of unnecessary overhead.


In [ ]:
# Exercise 7.4: STOP AND TRACE -- pytest fixture scope
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: Staging vs Production --
STAGING = {'model_stage': 'Staging', 'shadow_mode': True,
           'traffic_fraction': 0.0, 'alert_threshold': 'relaxed'}
PRODUCTION = {'model_stage': 'Production', 'shadow_mode': False,
              'traffic_fraction': 1.0, 'alert_threshold': 'strict'}
PROMOTION_CRITERIA = [
    'staging_auc >= production_auc - 0.01',
    'staging_p99_latency < 200ms',
    '7 days shadow mode with no anomalies',
    'manual sign-off from Dr. Emeka Obi',
]


### Expected Output (illustrative)

```
scope='module': the fixture runs ONCE per test module, not once per test function.
```
With model loading taking ~3 seconds and 20 tests in the module, `scope='module'` costs ~3
seconds total; `scope='function'` (the pytest default) would cost ~60 seconds — for a fixture
this expensive, module scope is the right call as long as no test mutates the shared model
object.



---

## FRIDAY -- "The Friday Build: Block a Deployment"


Introduce a deliberate test failure: reverse the directional assertion so it asserts delayed payers have LOWER default probability than on-time payers. Push to a PR. Verify GitHub Actions blocks the merge. Then fix the assertion and verify it passes.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: Block a Deployment --
# Step 1: introduce failure
# In tests/model/test_model.py, change:
# assert p_late > p_ontime
# to:
# assert p_late < p_ontime  # WRONG -- reversed direction

# Step 2: push to PR and observe GitHub Actions fail:
# AssertionError: Delayed payer (0.78) not less than on-time (0.24)
# PR merge is blocked

# Step 3: fix the assertion, push, verify the PR passes
# The fix proves the CI/CD pipeline correctly blocks broken models


### Expected Output (illustrative — GitHub Actions, deliberate failure)

```
✗ model_tests  FAILED
  AssertionError: Delayed payer (0.78) not less than on-time (0.82)
  Merge blocked: 1 required check failed
```
Then revert the deliberate reversal and push again:
```
✓ model_tests  PASSED
All checks have passed. Merge allowed.
```
If GitHub Actions doesn't actually block the merge on the first push, your branch-protection
rule — not the test — is what's misconfigured.


### Friday Workplace Moment

*DataFlow -- Friday standup.*

**Dr. Emeka Obi:** Did the deliberate failure block the deployment?

**Temi Adeyemi:** Yes. GitHub Actions failed model_tests: 'AssertionError: Delayed payer (0.78) not less than on-time (0.24).' PR merge was blocked.

**Sola Fashola:** And the data tests?

**Temi Adeyemi:** Data tests passed. model_tests ran after and caught the failure. The failure message named the exact values.

**Dr. Emeka Obi:** That is the pipeline working correctly. Data tests gate model tests. Model tests gate deployment.



---

## 🎁 BONUS EXERCISE — Pull Request Review Simulation

*Not required for the core path. Recommended for practice reading and writing PR reviews.*

A teammate has opened a PR against `main` with the diff below. Review it as if you were
Sola Fashola — leave comments, then decide: approve, request changes, or block?

```diff
--- a/tests/model/test_model.py
+++ b/tests/model/test_model.py
@@ -12,7 +12,7 @@ def test_pay0_directionality(model, holdout):
     p_late    = model.predict_proba(late_payer)[0][1]
     p_ontime  = model.predict_proba(on_time_payer)[0][1]
-    assert p_late > p_ontime
+    assert p_late > p_ontime - 0.05   # allow small tolerance for noise
@@ -30,6 +30,7 @@ def test_class_balance(train_df):
     rate = train_df['DEFAULT_NEXT_MONTH'].mean()
-    assert 0.10 <= rate <= 0.40
+    assert 0.05 <= rate <= 0.50   # widened range, was failing on new segment
```

**Your task — write actual PR review comments for both hunks:**
1. The first hunk weakens a *behavioural* test by adding a tolerance band. Is this
   loosening a real test until it passes, or a legitimate refinement? Justify either way.
2. The second hunk widens a *data* test's acceptable range in response to a failure. What
   question would you ask the author before approving this, given Week 3's root-cause lesson
   about investigating rather than adjusting the test?
3. Write your verdict: **Approve**, **Request Changes**, or **Block** — with the one-sentence
   reason you'd post as the top-level PR comment.

**Common mistake to watch for:** a test that fails is sometimes telling you the *code* is wrong
and sometimes telling you the *test* was too strict — but "the test failed" is never by itself
a reason to loosen the test. That decision needs the same evidence-based reasoning Week 3's
root-cause analysis required.


## YOUR WEEK 7 CHECKLIST

Check each item before moving on.

- [ ] Great Expectations suite covers schema, range, null, and class balance.
- [ ] Three behavioural tests encode domain knowledge that cannot be auto-generated.
- [ ] GitHub Actions blocks merges on any test failure, with data_tests gating model_tests.
- [ ] I can explain why staging and production must be separate environments.
- [ ] I blocked a deployment with a failing test and unblocked it with a fix.


## Challenge Task

> **Core path:** Implement property-based testing with Hypothesis: verify predictions are monotonic with respect to PAY_0. What percentage of randomly generated inputs satisfy monotonicity?

> **Advanced path:** Add a fairness test to CI/CD: approval rate for SEX=1 and SEX=2 must not differ by more than 5pp. Block deployment if it fails.


## Common Mistakes

**Only testing aggregate performance:** A model passing AUC >= 0.80 can still fail directionally. Behavioural tests catch what aggregate metrics hide.

**Running model tests before data tests:** A model test on corrupt data is meaningless and may pass for the wrong reason.

**Happy-path-only tests:** Test dangerous failure modes, not common inputs.



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)